In [11]:
# ── Step 1 · Download the raw Perturb-seq data ─────────────────────────────
#
# CausalBench ships with a broken downloader: it uses gdown (Google Drive
# only) against Figshare URLs, AND the URLs it points at sit behind an
# AWS-WAF challenge. The result is silent 0-byte files in causalbench_data/.
#
# `causalbench_loader.download_raw_data` replaces it with a plain urllib
# download against the working host (ndownloader.figshare.com), with two
# important properties:
#   • resumable — if a partial file is on disk it sends an HTTP Range
#     request for the remaining bytes (so a Ctrl-C and re-run continues
#     where you left off)
#   • size-validated — it cross-checks the final file against the known
#     Figshare file size, so a truncated file fails loudly instead of
#     blowing up later inside h5py
#
# Sizes: K562 = 10.6 GB, RPE1 = 8.7 GB. Total ~19 GB. Expect this to take
# a while on the first run; subsequent runs are no-ops.

DATA_DIR = "./causalbench_data"

from causalbench_loader import download_raw_data

download_raw_data(DATA_DIR)   # downloads BOTH datasets; pass files=["k562.h5ad"] for one

  ✓ k562.h5ad already complete (10.66 GB)
  ✓ rpe1.h5ad already complete (8.70 GB)


In [12]:
# RUNS SLOW

# ── Step 2 · Preprocess the raw h5ad into a cached .npz ────────────────────
#
# CausalBench's preprocessing pipeline (it lives in
# causalscbench.data_access.utils.preprocessing.preprocess_dataset):
#
#   1. scanpy reads the h5ad
#   2. sc.pp.normalize_per_cell  (counts per cell normalised, raw UMI
#      counts retained on .obs as UMI_count)
#   3. sc.pp.log1p
#   4. drop genes whose intervention appears in < 100 cells; those cells
#      get their intervention relabeled to "excluded"
#   5. keep only the gene columns that correspond to a perturbation we
#      actually intervened on
#
# The output is cached as causalbench_data/dataset_<line>.npz, so this
# whole step is a no-op on subsequent runs.

DATASET = "weissmann_k562"   # or "weissmann_rpe1"

from causalbench_loader import preprocess

npz_path = preprocess(DATA_DIR, DATASET)
print("Preprocessed dataset →", npz_path)

Preprocessed dataset → ./causalbench_data/dataset_k562.npz


In [13]:
# ── Step 3 · Apply the training regime and unpack the arrays ───────────────
#
# Internally this calls DatasetSplitter, which does a stratified 80/20
# train/test split (the 20% is held out; we work with the 80%). Then the
# regime selects which cells to keep:
#   • "observational"          → control cells only ("non-targeting")
#   • "partial_interventional" → control + perturbations on a random
#                                fraction of the perturbed genes
#   • "full_interventional"    → control + perturbations on ALL genes
#
# subset_data < 1.0 sub-samples the training data — useful for iterating
# fast before the full ~10^5-cell run.

REGIME = "observational"   # "observational" | "partial_interventional" | "full_interventional"
SUBSET = 1.0
PARTIAL_FRACTION = 0.5     # only used when REGIME == "partial_interventional"

from causalbench_loader import load_split

expression_matrix, interventions, gene_names = load_split(
    npz_path,
    regime=REGIME,
    subset_data=SUBSET,
    partial_fraction=PARTIAL_FRACTION,
)

# Shapes:
#   expression_matrix : np.ndarray (n_cells, n_genes)   log1p-normalised counts
#   interventions     : list[str], length n_cells       gene knocked out per cell,
#                                                       "non-targeting" for control
#   gene_names        : list[str], length n_genes
n_cells, n_genes = expression_matrix.shape
print(f"{n_cells:,} cells × {n_genes:,} genes")
print(f"unique interventions: {len(set(interventions))}")

8,553 cells × 1,158 genes
unique interventions: 1


In [14]:
import lingam #slow to load so do it seperately at the start

In [ ]:
# ── Step 4 · Toy LiNGAM run on the first 100 rows ──────────────────────────
#
# DirectLiNGAM (Shimizu et al., 2011) assumes the data is generated by a
# linear non-Gaussian acyclic structural equation model and recovers a
# unique causal ordering of the variables. It scales as O(n · p^3) and,
# more importantly, the estimator needs n >> p to be meaningful.
#
# Our slice is 100 cells × ~700 genes — n << p, so we further restrict to
# the 10 highest-variance genes. Just a test of the LiNGAM model

import numpy as np
import pandas as pd

N_ROWS = 100
N_GENES = 10

X_full = expression_matrix[:N_ROWS]                        # first 100 cells
gene_var = X_full.var(axis=0)
top_idx = np.argsort(gene_var)[-N_GENES:][::-1]            # highest-variance genes
X = X_full[:, top_idx]
selected_genes = [gene_names[i] for i in top_idx]

print(f"Fitting DirectLiNGAM on X shape {X.shape} (cells × genes)")
print(f"Genes:  {selected_genes}\n")

model = lingam.DirectLiNGAM()
model.fit(X)

# causal_order_ — indices into X's columns, in inferred causal order
# (earliest cause first).
print("Causal order (cause → effect):")
for rank, j in enumerate(model.causal_order_):
    print(f"  {rank+1:>2}. {selected_genes[j]}")

# adjacency_matrix_ — A[i, j] is the linear effect of gene j on gene i
# (rows are effects, columns are causes).
adj = pd.DataFrame(
    model.adjacency_matrix_,
    index=selected_genes,    # effect
    columns=selected_genes,  # cause
)
print("\nAdjacency matrix (rows = effect, columns = cause):")
print(adj.round(3).to_string())

Fitting DirectLiNGAM on X shape (100, 10) (cells × genes)
Genes:  [np.str_('ENSG00000134057'), np.str_('ENSG00000170312'), np.str_('ENSG00000088325'), np.str_('ENSG00000044574'), np.str_('ENSG00000092853'), np.str_('ENSG00000034510'), np.str_('ENSG00000150991'), np.str_('ENSG00000117399'), np.str_('ENSG00000114686'), np.str_('ENSG00000113460')]

Causal order (cause → effect):
   1. ENSG00000113460
   2. ENSG00000114686
   3. ENSG00000117399
   4. ENSG00000150991
   5. ENSG00000134057
   6. ENSG00000170312
   7. ENSG00000088325
   8. ENSG00000034510
   9. ENSG00000044574
  10. ENSG00000092853

Adjacency matrix (rows = effect, columns = cause):
                 ENSG00000134057  ENSG00000170312  ENSG00000088325  ENSG00000044574  ENSG00000092853  ENSG00000034510  ENSG00000150991  ENSG00000117399  ENSG00000114686  ENSG00000113460
ENSG00000134057            0.000            0.000              0.0              0.0              0.0              0.0            0.000            0.553            